In [2]:
# === Cell 1: Setup & load pre-LLM dataset ===
import os
import json
import time
import re
from pathlib import Path

import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
)

# Load the pre-LLM dataset assembled by 02_collect_data.ipynb
INPUT_PATH = Path("data/raw/dataset_pre_llm.csv")
df = pd.read_csv(INPUT_PATH, encoding="utf-8-sig")

print(f"Loaded {len(df)} rows from {INPUT_PATH}")
print(f"Columns: {list(df.columns)}")

# How many rows have non-empty release_notes (i.e., LLM can actually work on them)?
has_notes_mask = df["release_notes"].fillna("").str.strip().str.len() > 0
print(f"Rows with release_notes: {has_notes_mask.sum()} / {len(df)}")

Loaded 90 rows from data\raw\dataset_pre_llm.csv
Columns: ['app_name', 'platform', 'user_category', 'developer', 'store_category', 'version', 'release_date', 'initial_release', 'is_current', 'release_notes', 'source_url', 'data_quality_note']
Rows with release_notes: 45 / 90


In [4]:
# === Cell 2: Closed-set taxonomy of update categories ===
#
# These match the categories Prof. Bian listed in the task brief, lightly
# normalized to snake_case for stable downstream variable names.
# We use this exact set in the LLM prompt and to validate model outputs;
# any label outside this set is rejected and downgraded to "other".

UPDATE_CATEGORIES = {
    "bug_fixes_performance": "Crash fixes, speed improvements, stability",
    "ui_design":             "Visual redesign, layout changes, dark mode, icons",
    "privacy_data_policy":   "Permissions, tracking, GDPR/CCPA, data sharing changes",
    "ai_features":           "ML, AI assistants, generative features, AI-powered recommendations",
    "payments_monetization": "Subscriptions, in-app purchases, ads, pricing",
    "personalization_recs":  "Non-AI personalization, preferences, feed tuning",
    "security_account":      "Login, 2FA, account safety, encryption",
    "sdk_api_integration":   "Developer SDK, API, third-party integrations",
    "new_product_feature":   "Substantive new user-facing functionality",
    "other":                 "Use only if nothing else clearly fits",
}

# Reserved tag for rows where the release_notes field is empty/missing.
# Not part of the LLM's output space — applied only by post-hoc backfill.
UNAVAILABLE_TAG = "unavailable"

print(f"Defined {len(UPDATE_CATEGORIES)} categories:")
for k, v in UPDATE_CATEGORIES.items():
    print(f"  {k:<28}{v}")

Defined 10 categories:
  bug_fixes_performance       Crash fixes, speed improvements, stability
  ui_design                   Visual redesign, layout changes, dark mode, icons
  privacy_data_policy         Permissions, tracking, GDPR/CCPA, data sharing changes
  ai_features                 ML, AI assistants, generative features, AI-powered recommendations
  payments_monetization       Subscriptions, in-app purchases, ads, pricing
  personalization_recs        Non-AI personalization, preferences, feed tuning
  security_account            Login, 2FA, account safety, encryption
  sdk_api_integration         Developer SDK, API, third-party integrations
  new_product_feature         Substantive new user-facing functionality
  other                       Use only if nothing else clearly fits


In [5]:
# === Cell 3: Design the classification prompt ===
#
# Design philosophy (informed by inspecting our actual 45 release notes):
#
# 1. Closed-set categories (taxonomy from Cell 2) — no invented labels.
# 2. Multi-label allowed: a single update can be both "ui_design" and
#    "privacy_data_policy".
# 3. Few-shot examples chosen to mirror the THREE patterns we observed in
#    the real data:
#      - "Generic bug fix" (12/45 rows are this)
#      - "Specific feature update" (the typical case)
#      - "Marketing prose with little concrete content" (DoorDash-style)
# 4. Summary field: 5-15 words, present tense, NO marketing language.
#    Goal is a stem that can be tabulated as a research variable.
# 5. Confidence flag: lets us audit which rows the LLM was unsure about.
# 6. The LLM is told to extract signal even from buried mentions
#    (e.g., "personalized recommendations" inside an Airbnb marketing
#    paragraph should still trigger personalization_recs).

PROMPT_TEMPLATE = """You are a research assistant categorizing mobile app release notes for an academic study on data privacy, AI adoption, and digital markets.

Your task: classify ONE release note into a fixed set of categories and produce a neutral, research-ready summary.

═══════════════════════════════════════════════════════════════
CATEGORY DEFINITIONS (use these EXACT keys; do not invent new ones)
═══════════════════════════════════════════════════════════════
- bug_fixes_performance : Crash fixes, speed/stability improvements
- ui_design             : Visual redesign, layout, dark mode, icons
- privacy_data_policy   : Permissions, tracking changes, GDPR/CCPA, data sharing
- ai_features           : ML, AI assistants, generative features, AI-powered recommendations
- payments_monetization : Subscriptions, in-app purchases, ads, pricing
- personalization_recs  : Non-AI personalization, preferences, feed tuning, recommendations (when not explicitly AI-driven)
- security_account      : Login, 2FA, account safety, encryption
- sdk_api_integration   : Developer SDK, API, third-party integrations
- new_product_feature   : Substantive new user-facing functionality
- other                 : Use ONLY if nothing above clearly fits

═══════════════════════════════════════════════════════════════
RULES
═══════════════════════════════════════════════════════════════
1. Output ONE valid JSON object. No prose before or after.
2. "categories": list of 1-3 keys from the set above. Multiple labels are fine when warranted.
3. "summary": 5-15 words, present tense, neutral.
   - NO marketing language ("amazing", "delight", "smarter", "love", etc.)
   - NO corporate slogans or CEO quotes
   - State only what changed, in research-coding style
4. "confidence": "high" / "medium" / "low"
   - "low" = vague text where you're unsure what actually changed
5. If the note is just generic ("bug fixes and improvements" or similar with NO specifics):
   - categories: ["bug_fixes_performance"]
   - summary: "Generic bug fixes; no specific changes disclosed"
   - confidence: "high" (you're confidently saying it's vague)
6. Extract signal even from marketing prose. If a long paragraph mentions
   "personalized recommendations" or "new privacy controls" buried in CEO
   talk, surface those signals as labels.
7. CRITICAL: Classify based on the TEXT of the release note ALONE. 
   Even if you know the company is famous for AI/privacy/etc., 
   do NOT add categories that the actual text doesn't mention 
   or strongly imply. The app name is provided ONLY for resolving 
   ambiguous pronouns and product references; it is NOT evidence 
   for any category label.

═══════════════════════════════════════════════════════════════
EXAMPLES
═══════════════════════════════════════════════════════════════

Example 1 — Generic bug-fix:
Input: "Minor fixes and improvements"
Output: {"categories": ["bug_fixes_performance"], "summary": "Generic bug fixes; no specific changes disclosed", "confidence": "high"}

Example 2 — Specific feature with privacy element:
Input: "We've redesigned the home tab and added Apple Sign In for faster, more private login."
Output: {"categories": ["ui_design", "security_account", "privacy_data_policy"], "summary": "Home tab redesigned; Apple Sign In added for private login", "confidence": "high"}

Example 3 — Marketing prose with buried signal:
Input: "Our mission is to deliver good by connecting people and possibility. We're launching a new visual identity that reflects our spirit. Thank you for making us a success."
Output: {"categories": ["ui_design", "other"], "summary": "Visual rebrand; mostly marketing prose with no functional changes", "confidence": "medium"}

Example 4 — Personalization buried in marketing:
Input: "Now you can book homes, experiences, and services—all in one app. Our redesigned app includes personalized recommendations of unique places to stay."
Output: {"categories": ["ui_design", "personalization_recs", "new_product_feature"], "summary": "App redesign unifies booking; adds personalized recommendations", "confidence": "high"}

Example 5 — App tracking compliance:
Input: "Updates to comply with new App Tracking Transparency requirements."
Output: {"categories": ["privacy_data_policy"], "summary": "Updates for App Tracking Transparency compliance", "confidence": "high"}

═══════════════════════════════════════════════════════════════
NOW CLASSIFY THIS RELEASE NOTE
═══════════════════════════════════════════════════════════════
App: {APP_NAME} ({PLATFORM})
Release note:
\"\"\"
{RELEASE_NOTE}
\"\"\"

Output ONLY the JSON object."""

print("Prompt template length:", len(PROMPT_TEMPLATE), "chars")
print(f"≈ {len(PROMPT_TEMPLATE) // 4} input tokens per call (excluding the actual release note)")

Prompt template length: 4590 chars
≈ 1147 input tokens per call (excluding the actual release note)


In [6]:
# === Cell 4: Classifier function with validation, retry, and a 1-row sanity test ===
#
# Wraps the LLM call with:
#   - JSON parsing (defensive: tolerates leading/trailing text)
#   - Schema validation (categories must be from UPDATE_CATEGORIES; reject otherwise)
#   - Exponential-backoff retry on API errors
#   - Graceful fallback to {"other"} with confidence "low" if everything fails

def classify_release_note(
    release_note: str,
    app_name: str,
    platform: str,
    max_retries: int = 3,
) -> dict:
    """Classify ONE release note. Always returns a dict with the same schema."""
    if not release_note or not release_note.strip():
        return {
            "categories": [UNAVAILABLE_TAG],
            "summary": "Release notes not available for this version",
            "confidence": "high",
            "raw_response": None,
        }

    prompt = (
        PROMPT_TEMPLATE
        .replace("{APP_NAME}", app_name)
        .replace("{PLATFORM}", platform)
        .replace("{RELEASE_NOTE}", release_note.strip())
    )

    last_error = None
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="deepseek-chat",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=300,
                temperature=0.0,  # Deterministic — same input -> same output
            )
            raw = response.choices[0].message.content.strip()

            # Defensive JSON extraction (handles minor formatting drift)
            start = raw.find("{")
            end = raw.rfind("}") + 1
            if start == -1 or end == 0:
                raise ValueError("No JSON object found in response")
            parsed = json.loads(raw[start:end])

            # Schema validation
            categories = parsed.get("categories", [])
            if not isinstance(categories, list) or not categories:
                raise ValueError(f"Invalid 'categories' field: {categories!r}")

            valid_keys = set(UPDATE_CATEGORIES.keys())
            cleaned_cats = [c for c in categories if c in valid_keys]
            if not cleaned_cats:
                cleaned_cats = ["other"]

            summary = str(parsed.get("summary", "")).strip()
            confidence = parsed.get("confidence", "medium")
            if confidence not in {"high", "medium", "low"}:
                confidence = "medium"

            return {
                "categories":   cleaned_cats,
                "summary":      summary,
                "confidence":   confidence,
                "raw_response": raw,
            }

        except (json.JSONDecodeError, ValueError, KeyError) as e:
            last_error = e
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
        except Exception as e:
            last_error = e
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)

    # Final fallback — never raise; downstream code expects a dict
    return {
        "categories": ["other"],
        "summary": f"Classification failed: {type(last_error).__name__}",
        "confidence": "low",
        "raw_response": None,
    }


# === Sanity test: classify ONE row before launching batch ===
test_row = df[df["release_notes"].fillna("").str.strip().str.len() > 0].iloc[0]
print(f"Testing on:\n  {test_row['app_name']} {test_row['platform']} v{test_row['version']}")
print(f"  Release note: {test_row['release_notes'][:150]}...\n")

result = classify_release_note(
    test_row["release_notes"],
    test_row["app_name"],
    test_row["platform"],
)
print("LLM result:")
for k, v in result.items():
    if k != "raw_response":
        print(f"  {k}: {v}")

Testing on:
  Airbnb iOS v26.19
  Release note: Now you can book homes, experiences, and services—all in one app. Our redesigned app includes personalized recommendations of unique places to stay, u...

LLM result:
  categories: ['new_product_feature', 'ui_design', 'personalization_recs']
  summary: Unified booking for homes, experiences, services; redesigned app with personalized recommendations
  confidence: high


In [7]:
# === Cell 4.1: Two more sanity tests on edge cases ===

edge_cases = [
    # Case 1: pure "bug fixes and improvements" — checks Rule 5 in prompt
    {
        "app": "ChatGPT",
        "platform": "Android",
        "notes": "Minor fixes and improvements"
    },
    # Case 2: marketing prose — checks ability to extract buried signal
    {
        "app": "DoorDash",
        "platform": "Android",
        "notes": (
            "Our mission is to deliver good by connecting people and possibility. "
            "If we can play a small role in helping you spend more time with your "
            "friends and family or get ahead on your favorite projects, then we "
            "have delivered good. We're launching a new set of initiatives to "
            "deliver good within our communities, and you'll see a new visual "
            "identity that reflects our spirit."
        )
    },
]

for case in edge_cases:
    print(f"\n--- {case['app']} ({case['platform']}) ---")
    print(f"Input: {case['notes'][:120]}{'...' if len(case['notes']) > 120 else ''}")
    result = classify_release_note(case["notes"], case["app"], case["platform"])
    print(f"\n  categories: {result['categories']}")
    print(f"  summary:    {result['summary']}")
    print(f"  confidence: {result['confidence']}")


--- ChatGPT (Android) ---
Input: Minor fixes and improvements

  categories: ['bug_fixes_performance']
  summary:    Generic bug fixes; no specific changes disclosed
  confidence: high

--- DoorDash (Android) ---
Input: Our mission is to deliver good by connecting people and possibility. If we can play a small role in helping you spend mo...

  categories: ['ui_design', 'other']
  summary:    Visual rebrand; community initiatives announced; no functional changes
  confidence: medium


In [8]:
# === Cell 5: Batch-classify all rows with release_notes ===
#
# Design notes:
# - Only call the API for rows that actually have release_notes text.
#   Empty rows are handled by Cell 6 (post-hoc backfill with "unavailable").
# - Save partial results to disk every CHECKPOINT_EVERY rows so a network
#   interruption doesn't cost us all progress.
# - Skip rows already classified in a prior partial run, so re-running
#   this cell is idempotent (won't waste API calls).
# - Polite 0.3s sleep between calls to avoid rate-limit spikes.

CHECKPOINT_PATH = Path("data/raw/llm_results_partial.csv")
CHECKPOINT_EVERY = 10

# Load any prior partial run so we can resume
if CHECKPOINT_PATH.exists():
    prior_results = pd.read_csv(CHECKPOINT_PATH, encoding="utf-8-sig")
    completed_keys = set(zip(prior_results["app_name"], prior_results["platform"], prior_results["version"]))
    print(f"Found {len(prior_results)} prior results. Will skip these.\n")
    results_so_far = prior_results.to_dict("records")
else:
    completed_keys = set()
    results_so_far = []

# Identify rows that need classification: have release_notes AND not yet done
notes_mask = df["release_notes"].fillna("").str.strip().str.len() > 0
to_classify = df[notes_mask].copy()

# Filter out rows already classified
def row_key(row):
    return (row["app_name"], row["platform"], row["version"])

pending = to_classify[~to_classify.apply(lambda r: row_key(r) in completed_keys, axis=1)]
print(f"Total rows with release_notes: {len(to_classify)}")
print(f"Already classified: {len(completed_keys)}")
print(f"Still to classify:  {len(pending)}\n")

# Run batch
total = len(pending)
for i, (idx, row) in enumerate(pending.iterrows(), start=1):
    app_name = row["app_name"]
    platform = row["platform"]
    version  = row["version"]
    notes    = row["release_notes"]

    print(f"[{i:2d}/{total}] {app_name:18s} {platform:7s} v{version}", end=" ")
    result = classify_release_note(notes, app_name, platform)

    results_so_far.append({
        "app_name":   app_name,
        "platform":   platform,
        "version":    version,
        "categories": json.dumps(result["categories"]),     # store as JSON string for CSV
        "llm_summary": result["summary"],
        "confidence": result["confidence"],
    })
    print(f"-> {result['categories']}  ({result['confidence']})")

    # Periodic checkpoint
    if i % CHECKPOINT_EVERY == 0 or i == total:
        pd.DataFrame(results_so_far).to_csv(CHECKPOINT_PATH, index=False, encoding="utf-8-sig")
        print(f"    [checkpoint saved: {len(results_so_far)} total rows]")

    time.sleep(0.3)  # Light pacing — avoid rate-limit spikes

# Final save
llm_results_df = pd.DataFrame(results_so_far)
llm_results_df.to_csv(CHECKPOINT_PATH, index=False, encoding="utf-8-sig")
print(f"\n=== Done. {len(llm_results_df)} rows classified. ===")
print(f"\nCategory distribution (multi-label, so totals exceed row count):")
all_cats = []
for cats_json in llm_results_df["categories"]:
    all_cats.extend(json.loads(cats_json))
print(pd.Series(all_cats).value_counts())

print(f"\nConfidence distribution:")
print(llm_results_df["confidence"].value_counts())

Total rows with release_notes: 45
Already classified: 0
Still to classify:  45

[ 1/45] Airbnb             iOS     v26.19 -> ['new_product_feature', 'ui_design', 'personalization_recs']  (high)
[ 2/45] Amazon Shopping    Android v32.1.2.100 -> ['new_product_feature', 'bug_fixes_performance']  (high)
[ 3/45] Amazon Shopping    Android v32.3.0.100 -> ['new_product_feature', 'bug_fixes_performance']  (high)
[ 4/45] Amazon Shopping    Android v32.5.0.100 -> ['new_product_feature', 'bug_fixes_performance']  (high)
[ 5/45] Amazon Shopping    Android v32.6.0.100 -> ['new_product_feature', 'bug_fixes_performance']  (high)
[ 6/45] Amazon Shopping    Android v32.8.0.100 -> ['new_product_feature', 'bug_fixes_performance']  (high)
[ 7/45] Amazon Shopping    Android v32.9.0.100 -> ['new_product_feature', 'bug_fixes_performance']  (high)
[ 8/45] Amazon Shopping    Android v32.9.0.100 -> ['new_product_feature', 'bug_fixes_performance']  (high)
[ 9/45] Amazon Shopping    iOS     v27.9.0 -> ['ui_design

In [9]:
# === Cell 6: Backfill rows without release_notes ===
#
# 45 of our 90 rows have no release_notes (Wayback parse misses, Cloudflare
# blocks on APKMirror detail pages, etc.). For these we apply a uniform
# "unavailable" tag so the final spreadsheet has every row labeled —
# educators can still tabulate which app/platform/version pairs lacked
# release_notes data.

unavailable_rows = []
classified_keys = set(zip(
    llm_results_df["app_name"],
    llm_results_df["platform"],
    llm_results_df["version"],
))

for _, row in df.iterrows():
    key = (row["app_name"], row["platform"], row["version"])
    if key in classified_keys:
        continue
    # This row has no release_notes (or wasn't classified)
    unavailable_rows.append({
        "app_name":   row["app_name"],
        "platform":   row["platform"],
        "version":    row["version"],
        "categories": json.dumps([UNAVAILABLE_TAG]),
        "llm_summary": "Release notes not available for this version",
        "confidence": "high",
    })

unavailable_df = pd.DataFrame(unavailable_rows)
print(f"Backfilled {len(unavailable_df)} rows with 'unavailable' tag")

# Combine LLM-classified + unavailable
all_labels_df = pd.concat([llm_results_df, unavailable_df], ignore_index=True)
print(f"Total labeled rows: {len(all_labels_df)} (should equal {len(df)})")

Backfilled 44 rows with 'unavailable' tag
Total labeled rows: 89 (should equal 90)


In [10]:
# === Cell 7: Merge LLM labels back into the main dataset ===
#
# Join on (app_name, platform, version). The result is the FINAL dataset
# that satisfies all 13 fields in Prof. Bian's task brief.

# Re-load the main dataset to get a clean reference
main_df = pd.read_csv(INPUT_PATH, encoding="utf-8-sig")

# Merge — left join so we keep every original row
merged = main_df.merge(
    all_labels_df,
    on=["app_name", "platform", "version"],
    how="left",
)

# Sanity: every row should have labels now
missing_labels = merged["categories"].isna().sum()
print(f"Rows still missing labels after merge: {missing_labels}")
if missing_labels > 0:
    print("WARNING: some rows didn't get labels — check for join key mismatches")
    print(merged[merged["categories"].isna()][["app_name", "platform", "version"]])

# Reorder columns to match the 13 fields Prof. Bian asked for
final_cols = [
    "app_name",            # 1. App name
    "platform",            # 2. Platform
    "developer",           # 3. Developer
    "store_category",      # 4. App category (store-reported)
    "user_category",       #    + our top-level grouping for cross-reference
    "version",             # 5. Version number
    "release_date",        # 6. Version release/update date
    "is_current",          # 7. Whether this is the current version
    "initial_release",     # 8. Initial app release date
    "release_notes",       # 9. Update description / release notes
    "categories",          # 10. Standardized update categories (LLM)
    "llm_summary",         # 11. Standardized summary (LLM)
    "confidence",          #     + LLM self-rated confidence (auditing aid)
    "source_url",          # 12. Source URL
    "data_quality_note",   # 13. Notes on data quality
]
final = merged[final_cols].sort_values(
    ["app_name", "platform", "release_date"], na_position="last"
).reset_index(drop=True)

# Save the canonical analysis-ready CSV
OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
final.to_csv(OUTPUT_DIR / "app_updates_labeled.csv", index=False, encoding="utf-8-sig")

print(f"\nFinal dataset: {len(final)} rows × {len(final.columns)} cols")
print(f"Saved to {OUTPUT_DIR / 'app_updates_labeled.csv'}")
final.head(3)

Rows still missing labels after merge: 0

Final dataset: 98 rows × 15 cols
Saved to data\processed\app_updates_labeled.csv


,app_name,platform,developer,store_category,user_category,version,release_date,is_current,initial_release,release_notes,categories,llm_summary,confidence,source_url,data_quality_note
0,Airbnb,Android,Airbnb,Travel & Local,Travel,26.18,2026-05-05,True,NaN,NaN,"[""unavailable""]",Release notes not available for this version,high,https://play.google.com/store/apps/details?id=...,Current version retrieved from google-play-scr...
1,Airbnb,iOS,"Airbnb, Inc.",Travel,Travel,26.19,2026-05-06,True,2010-11-10T20:28:11Z,"Now you can book homes, experiences, and servi...","[""new_product_feature"", ""ui_design"", ""personal...","Unified booking for homes, experiences, servic...",high,https://itunes.apple.com/lookup?id=401626263,Current version retrieved from iTunes Lookup A...
2,Amazon Shopping,Android,Amazon Mobile LLC,Shopping,E-commerce,32.1.2.100,2026-01-10,False,NaN,Access popular pages quickly with our new shor...,"[""new_product_feature"", ""bug_fixes_performance""]","Added app icon shortcuts for Orders, Deals, Ca...",high,https://www.apkmirror.com/apk/amazon-mobile-ll...,Historical version from APKMirror (stratified ...


In [11]:
# === Cell 8: Export to Excel with summary sheet ===
#
# Two sheets:
#   1. "data" — the 90-row labeled dataset (one row per app-platform-version)
#   2. "methodology_summary" — the narrative summary Prof. Bian asked for
#      at the end of the spreadsheet (data sources, time-series patterns,
#      challenges). We seed it here with placeholders to be expanded
#      manually after running this cell.

OUTPUT_XLSX = Path("output/app_updates.xlsx")
OUTPUT_XLSX.parent.mkdir(parents=True, exist_ok=True)

# Build summary sheet skeleton — to be edited with real prose later
summary_lines = [
    ["App Update History — Methodology and Findings Summary"],
    [""],
    ["Author", "Chengyu Liu (UBC Statistics)"],
    ["Submission", "WLIURA S26 RA Test Task — Prof. Bo Bian"],
    ["Dataset rows", str(len(final))],
    ["Apps covered", ", ".join(sorted(final["app_name"].unique()))],
    ["Date range", f"{final['release_date'].min()} to {final['release_date'].max()}"],
    [""],
    ["1. METHODOLOGY"],
    ["[To be expanded — describe data sources, LLM classifier, GitHub link]"],
    [""],
    ["2. TIME-SERIES PATTERNS"],
    ["[To be expanded — discuss update frequency, category shifts over time]"],
    [""],
    ["3. CHALLENGES & DATA QUALITY"],
    ["[To be expanded — discuss Cloudflare blocks, Wayback parse rates, "
     "Android historical depth limitations, AI/privacy near-zero signal]"],
    [""],
    ["4. CATEGORY DISTRIBUTION"],
]

# Append category counts to summary sheet
all_cats_in_final = []
for cats_json in final["categories"].dropna():
    all_cats_in_final.extend(json.loads(cats_json))
cat_counts = pd.Series(all_cats_in_final).value_counts()
for cat, count in cat_counts.items():
    summary_lines.append([cat, int(count)])

summary_df = pd.DataFrame(summary_lines)

# Write workbook
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    final.to_excel(writer, sheet_name="data", index=False)
    summary_df.to_excel(writer, sheet_name="methodology_summary",
                        index=False, header=False)

print(f"Excel saved: {OUTPUT_XLSX}")
print(f"Sheets: 'data' ({len(final)} rows) + 'methodology_summary'")

Excel saved: output\app_updates.xlsx
Sheets: 'data' (98 rows) + 'methodology_summary'
